In [6]:
import os
import sys
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from datetime import date
plt.rcParams["figure.figsize"] = (24,18)

from ultralytics import YOLO
from cv_utils import *

import torch
torch.cuda.empty_cache()

### Load dataset and pretrained model

In [7]:
# Load original, pretrained model
model_path = os.path.join(os.path.dirname(os.getcwd()), 'models')
print(os.path.exists(os.path.join(model_path, "yolov8n.pt")))
model = YOLO(os.path.join(model_path, "yolov8n.pt"))

True


In [8]:
# Load YOLO dataset
data_path = os.path.join(os.path.dirname(os.getcwd()), 'data')
print(data_path)
dataset_name = 'cropped_all_match_yolov8'
dataset_path = os.path.join(data_path, 'coco_datasets/'+dataset_name)
metadata_path = os.path.join(dataset_path, 'data.yaml')

/media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data


### Model training

In [9]:
results = model.train(data=metadata_path, 
                      batch=4,
                      epochs=9000, 
                      imgsz=2048,
                      verbose=True, 
                      resume=False)

New https://pypi.org/project/ultralytics/8.0.229 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.0.149 🚀 Python-3.8.10 torch-1.13.1+cu116 CUDA:0 (NVIDIA GeForce RTX 3080, 9987MiB)
WARNING ⚠️ Upgrade to torch>=2.0.0 for deterministic training.
engine/trainer: task=detect, mode=train, model=/media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/models/yolov8n.pt, data=/media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/coco_datasets/cropped_all_match_yolov8/data.yaml, epochs=4500, patience=50, batch=4, imgsz=2048, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=None, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_d

KeyboardInterrupt: 

In [6]:
# Bring trained model to designated folder
result_folder = os.path.join(os.getcwd(), 'runs/detect')
saved_model_folder = os.path.join(result_folder, 'train14/weights/best.pt')
shutil.copy(saved_model_folder, os.path.join(model_path, 'best.pt'))

'/media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/models/best.pt'

### Inference

In [8]:
vid_name = 'Ars-MC 1st half'
inference_data_path = os.path.join(data_path, 'images/' + vid_name)

dest_path = os.path.join(data_path, 'annotated_frames' + '/' + vid_name)
if not os.path.exists(dest_path):
    os.mkdir(dest_path)


for image_name in os.listdir(inference_data_path):
    frame = os.path.join(inference_data_path, image_name)

    # Run YOLOv8 inference on the frame
    results = model(frame)

    # Visualize the results on the frame
    annotated_frame = results[0].plot(probs=False, font_size=6)

    # Display the annotated frame
    cv2.imwrite(os.path.join(dest_path, image_name), annotated_frame)

ref_image = cv2.imread(os.path.join(dest_path, os.listdir(dest_path)[0]), cv2.IMREAD_UNCHANGED)
h, w, _ = ref_image.shape
video= cv2.VideoWriter(os.path.join(dest_path, vid_name+'.mp4'), cv2.VideoWriter_fourcc(*'mp4v'), 3, (w,h))
for image_name in os.listdir(dest_path):
    frame = cv2.imread(os.path.join(dest_path, image_name), cv2.IMREAD_UNCHANGED)
    video.write(frame)
video.release()


image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/Ars-MC 1st half/000000000507.png: 1152x2048 15 persons, 1 ball, 8.5ms
Speed: 21.8ms preprocess, 8.5ms inference, 0.6ms postprocess per image at shape (1, 3, 1152, 2048)

image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/Ars-MC 1st half/000000000508.png: 1152x2048 16 persons, 8.5ms
Speed: 6.6ms preprocess, 8.5ms inference, 0.6ms postprocess per image at shape (1, 3, 1152, 2048)

image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/Ars-MC 1st half/000000000509.png: 1152x2048 16 persons, 7.6ms
Speed: 6.6ms preprocess, 7.6ms inference, 1.1ms postprocess per image at shape (1, 3, 1152, 2048)

image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/Ars-MC 1st half/000000000510.png: 1152x2048 16 persons, 8.5ms
Speed: 6.4ms preprocess, 8.5ms inference, 1.9ms postprocess per image at shape (1